# Notebook 3: Create Foundry Resources & Build Your First Agent
## From Resource Setup to Multi-Turn Agent Conversations

**Sources:**
- [Quickstart: Set up Microsoft Foundry resources](https://learn.microsoft.com/en-us/azure/foundry/tutorials/quickstart-create-foundry-resources?tabs=azurecli)
- [Microsoft Foundry Quickstart — Get Started with Code](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code?tabs=python)
- [Tutorial: Idea to Prototype — Build and Evaluate an Enterprise Agent](https://learn.microsoft.com/en-us/azure/foundry/tutorials/developer-journey-idea-to-prototype?tabs=python)
- [Ask AI for Help](https://learn.microsoft.com/en-us/azure/foundry/concepts/ask-ai)
- [Migrate from the Foundry (classic) Portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/navigate-from-classic)

---

This notebook covers:
1. Creating Foundry resources via Azure CLI
2. Deploying a model (gpt-4.1-mini)
3. Getting your project endpoint
4. Granting team access (RBAC)
5. Chatting with a model via the Responses API
6. Creating an agent with instructions
7. Multi-turn conversations with an agent
8. Enterprise agent patterns (SharePoint + MCP + Evaluation)
9. Ask AI — the portal AI assistant
10. Migrating from Foundry (classic)

## End-to-End Flow

```
┌─────────────────────────────────────────────────────────────────────────┐
│                     Developer Journey Overview                          │
│                                                                         │
│  ┌───────────────┐    ┌───────────────┐    ┌─────────────────────────┐  │
│  │ 1. CREATE      │    │ 2. DEPLOY     │    │ 3. CODE                 │  │
│  │ RESOURCES      │───▶│ A MODEL       │───▶│ Chat / Agent / Evaluate │  │
│  │                │    │               │    │                         │  │
│  │ • Resource Grp │    │ • gpt-4.1-mini│    │ • Responses API         │  │
│  │ • Foundry Res  │    │ • Standard SKU│    │ • Agent creation        │  │
│  │ • Project      │    │ • Verify      │    │ • Multi-turn chat       │  │
│  └───────────────┘    └───────────────┘    └─────────────────────────┘  │
│         │                                            │                  │
│         ▼                                            ▼                  │
│  ┌───────────────┐                          ┌─────────────────────────┐  │
│  │ 4. GRANT      │                          │ 5. ENTERPRISE PATTERNS  │  │
│  │ TEAM ACCESS   │                          │ SharePoint + MCP        │  │
│  │ (RBAC Roles)  │                          │ Batch Evaluation        │  │
│  └───────────────┘                          └─────────────────────────┘  │
└─────────────────────────────────────────────────────────────────────────┘
```

---
## Prerequisites

| Requirement | Description | Status |
|---|---|---|
| **Azure Subscription** | Active subscription ([Create free account](https://azure.microsoft.com/free/)) | [ ] |
| **Azure CLI 2.67+** | Check with `az version` ([Install](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli)) | [ ] |
| **Logged in** | Run `az login` before executing CLI commands | [ ] |
| **RBAC Role** | **Foundry Owner** or **Azure Account AI Owner** on subscription/resource group | [ ] |
| **Python 3.10+** | Python runtime | [ ] |
| **Packages** | `azure-ai-projects>=2.0.0`, `azure-identity`, `openai` | [ ] |

> **Note on RBAC roles:** The Foundry RBAC roles were recently renamed. **Foundry User**, **Foundry Owner**, **Foundry Account Owner**, and **Foundry Project Manager** were previously named Azure AI User, Azure AI Owner, Azure AI Account Owner, and Azure AI Project Manager. Role IDs and core permissions are unchanged.

In [ ]:
# Verify local prerequisites
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10 or higher is required!"
print("Python version check passed.")

import subprocess
result = subprocess.run(["az", "version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"Azure CLI installed: OK")
else:
    print("Azure CLI is NOT installed. Install from: https://learn.microsoft.com/en-us/cli/azure/install-azure-cli")

In [ ]:
# Install required packages
!pip install azure-ai-projects>=2.0.0 azure-identity openai python-dotenv

---
## Part 1: Create Foundry Resources via Azure CLI

This section walks through creating a Foundry resource, project, and deploying a model using the Azure CLI.

> **Portal alternative:** You can also do all of this in the [Microsoft Foundry portal](https://ai.azure.com). Sign in, toggle **New Foundry** on, and follow the guided project creation flow.

### Step 1.1 — Create a Resource Group

In [ ]:
# Step 1.1 — Create a resource group (or use an existing one)
# Replace values as needed for your environment

!az group create --name my-foundry-rg --location eastus

### Step 1.2 — Create the Foundry Resource

This creates a single **AIServices** resource with project management enabled. This replaces the old Hub + Azure OpenAI + Azure AI Services trio with a single unified resource.

In [ ]:
# Step 1.2 — Create the Foundry resource
# The --allow-project-management flag enables project creation within this resource
# The name must be globally unique — change "my-foundry-resource" if taken

!az cognitiveservices account create \
    --name my-foundry-resource \
    --resource-group my-foundry-rg \
    --kind AIServices \
    --sku s0 \
    --location eastus \
    --allow-project-management

### Step 1.3 — Set a Custom Subdomain

The custom domain name must be globally unique. This becomes part of your project endpoint URL.

In [ ]:
# Step 1.3 — Set a custom subdomain for the resource
!az cognitiveservices account update \
    --name my-foundry-resource \
    --resource-group my-foundry-rg \
    --custom-domain my-foundry-resource

### Step 1.4 — Create a Project

In [ ]:
# Step 1.4 — Create a project under the Foundry resource
!az cognitiveservices account project create \
    --name my-foundry-resource \
    --resource-group my-foundry-rg \
    --project-name my-foundry-project \
    --location eastus

In [ ]:
# Step 1.4b — Verify the project was created
!az cognitiveservices account project show \
    --name my-foundry-resource \
    --resource-group my-foundry-rg \
    --project-name my-foundry-project

---
## Part 2: Deploy a Model

Deploy **gpt-4.1-mini** (or any available model) so you can use it from code. This uses the Standard SKU with a capacity of 10.

In [ ]:
# Deploy a model
!az cognitiveservices account deployment create \
    --name my-foundry-resource \
    --resource-group my-foundry-rg \
    --deployment-name gpt-4.1-mini \
    --model-name gpt-4.1-mini \
    --model-version "2025-04-14" \
    --model-format OpenAI \
    --sku-capacity 10 \
    --sku-name Standard

In [ ]:
# Verify the deployment succeeded (look for "provisioningState": "Succeeded")
!az cognitiveservices account deployment show \
    --name my-foundry-resource \
    --resource-group my-foundry-rg \
    --deployment-name gpt-4.1-mini

---
## Part 3: Get Your Project Endpoint

Your **project endpoint** is the single URL you use to connect from code. It follows this format:

```
https://<resource_name>.ai.azure.com/api/projects/<project_name>
```

You can also find it in the [Foundry portal](https://ai.azure.com) > Your Project > **Overview** page (the "welcome screen").

### 3.1 Configure environment variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set your project endpoint — replace with your actual values
# Option 1: Set directly
# PROJECT_ENDPOINT = "https://my-foundry-resource.ai.azure.com/api/projects/my-foundry-project"

# Option 2: Load from .env file (recommended)
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "your_project_endpoint")
AGENT_NAME = os.getenv("AGENT_NAME", "MyAgent")

if PROJECT_ENDPOINT == "your_project_endpoint":
    print("WARNING: Please set PROJECT_ENDPOINT in your .env file or above.")
    print("Format: https://<resource_name>.ai.azure.com/api/projects/<project_name>")
else:
    print(f"Project endpoint: {PROJECT_ENDPOINT[:60]}...")
    print(f"Agent name: {AGENT_NAME}")

---
## Part 4: Grant Team Access (RBAC)

If you're administering a team, assign the **Foundry User** role to team members so they can use the project and deployed models.

| Role | GUID | Purpose |
|---|---|---|
| **Foundry User** | `53ca6127-db72-4b80-b1b0-d745d6d5456d` | Minimum permissions to build and test AI apps |
| **Foundry Owner** | `c883944f-8b7b-4483-af10-35834be79c4a` | Full control over the project |
| **Foundry Account Owner** | `e47c6f54-e4a2-4754-9501-8e0985b135e1` | Full control over the Foundry resource |
| **Foundry Project Manager** | `eadc314b-1a2d-4efa-be10-5d325db5065e` | Manage project settings and members |

> **Tip:** Use role GUIDs instead of names in scripts to avoid issues during the rename rollout.

In [ ]:
# Get the project's resource ID
# PROJECT_ID=$(az cognitiveservices account project show \
#     --name my-foundry-resource \
#     --resource-group my-foundry-rg \
#     --project-name my-foundry-project \
#     --query id -o tsv)

# Assign the Foundry User role to a team member
# Replace "user@contoso.com" with the actual email address
# !az role assignment create \
#     --role "53ca6127-db72-4b80-b1b0-d745d6d5456d" \
#     --assignee "user@contoso.com" \
#     --scope $PROJECT_ID

# To add a security group instead of an individual:
# !az role assignment create \
#     --role "53ca6127-db72-4b80-b1b0-d745d6d5456d" \
#     --assignee-object-id "<security-group-object-id>" \
#     --assignee-principal-type Group \
#     --scope $PROJECT_ID

print("RBAC commands are commented out — uncomment and fill in values to run.")

---
## Part 5: Chat with a Model (Responses API)

The basic building block — send an input, receive a response. This uses the new unified pattern:

1. Create an `AIProjectClient` from your project endpoint
2. Get an OpenAI-compatible client from it
3. Call the **Responses API** (replaces Chat Completions)

> **Important:** Code uses **Azure AI Projects 2.x** and is incompatible with Azure AI Projects 1.x.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Create project client — single entry point for all Foundry APIs
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Get an OpenAI-compatible client from the project
openai_client = project.get_openai_client()

# Run a Responses API call
response = openai_client.responses.create(
    model="gpt-4.1-mini",  # Supports all Foundry direct models
    input="What is the size of France in square miles?",
)

print(f"Response: {response.output_text}")

---
## Part 6: Create an Agent

An **agent** defines core behavior with a model and instructions. Once created, it ensures consistent responses without repeating instructions each time. Agents are versioned — you can update or delete them anytime.

Key concepts:
- **Agent Name** — identifier you reference when chatting
- **Agent Version** — auto-incremented with `create_version()`
- **PromptAgentDefinition** — defines model + instructions + optional tools

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

# Create project client
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Create an agent with a model and instructions
agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",  # Supports all Foundry direct models
        instructions="You are a helpful assistant that answers general questions clearly and concisely.",
    ),
)

print(f"Agent created!")
print(f"  Name:    {agent.name}")
print(f"  Version: {agent.version}")
print(f"  ID:      {agent.id}")

---
## Part 7: Multi-Turn Conversation with an Agent

Use a **conversation** to maintain history across interactions. The agent remembers context from previous turns — no need to re-send the full history yourself.

```
Turn 1: "What is the size of France?"  →  Agent responds with size
Turn 2: "And what is the capital?"     →  Agent knows you're still talking about France
```

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Create project and OpenAI clients
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai_client = project.get_openai_client()

# Create a conversation for multi-turn chat
conversation = openai_client.conversations.create()
print(f"Conversation created: {conversation.id}\n")

# Turn 1 — Ask a question
print("--- Turn 1 ---")
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="What is the size of France in square miles?",
)
print(f"Q: What is the size of France in square miles?")
print(f"A: {response.output_text}\n")

# Turn 2 — Ask a follow-up (the agent remembers context)
print("--- Turn 2 ---")
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    input="And what is the capital city?",
)
print(f"Q: And what is the capital city?")
print(f"A: {response.output_text}")

---
## Part 8: Enterprise Agent Patterns — Idea to Prototype

The [Developer Journey tutorial](https://learn.microsoft.com/en-us/azure/foundry/tutorials/developer-journey-idea-to-prototype?tabs=python) shows how to build a **Modern Workplace Assistant** that combines:

- **SharePoint** — internal company policies and documents
- **MCP (Model Context Protocol)** — external technical docs from Microsoft Learn
- **Batch Evaluation** — validate agent quality with built-in evaluators

### Architecture

```
┌───────────────────────────────────────────────────────────────┐
│              Modern Workplace Assistant                        │
│                                                               │
│  ┌─────────────────┐  ┌─────────────────┐  ┌──────────────┐  │
│  │  SharePoint Tool │  │  MCP Tool        │  │  Agent Core  │  │
│  │                  │  │                  │  │              │  │
│  │  Company policies│  │  Microsoft Learn │  │  gpt-4o-mini │  │
│  │  Internal docs   │  │  Technical docs  │  │  Instructions│  │
│  └────────┬─────────┘  └────────┬─────────┘  └──────┬───────┘  │
│           │                     │                    │          │
│           └─────────────────────┴────────────────────┘          │
│                              │                                  │
│                    ┌─────────▼──────────┐                       │
│                    │  Responses API     │                       │
│                    │  (Conversations)   │                       │
│                    └────────────────────┘                       │
└───────────────────────────────────────────────────────────────┘
                              │
                    ┌─────────▼──────────┐
                    │  Batch Evaluation   │
                    │  • builtin.violence │
                    │  • builtin.fluency  │
                    │  • builtin.task_    │
                    │    adherence        │
                    └────────────────────┘
```

### Get the sample code

In [ ]:
# Clone the enterprise agent tutorial sample (shallow clone)
!git clone --depth 1 https://github.com/microsoft-foundry/foundry-samples.git foundry-samples-tutorial

### Tutorial file structure

```
enterprise-agent-tutorial/
└── 1-idea-to-prototype/
    ├── .env                             # Local environment variables (create this)
    ├── evaluate.py                      # Business evaluation framework
    ├── evaluation_results.json
    ├── main.py                          # Modern Workplace Assistant
    ├── questions.jsonl                  # Business test scenarios (4 questions)
    ├── requirements.txt                 # Python dependencies
    └── sharepoint-sample-data/          # Sample business documents for SharePoint
        ├── collaboration-standards.docx
        ├── data-governance-policy.docx
        ├── remote-work-policy.docx
        └── security-guidelines.docx
```

### 8.1 Configure the `.env` for the enterprise tutorial

```dotenv
# Foundry configuration
FOUNDRY_PROJECT_ENDPOINT=https://<your-project>.aiservices.azure.com
FOUNDRY_MODEL_NAME=gpt-4o-mini

# Microsoft Learn MCP Server (optional)
MCP_SERVER_URL=https://learn.microsoft.com/api/mcp

# SharePoint integration (optional — requires connection name)
SHAREPOINT_CONNECTION_NAME=<your-sharepoint-connection-name>
```

### 8.2 Configure the SharePoint tool

The agent uses a `SharepointPreviewTool` to access company documents stored in SharePoint:

```python
from azure.ai.projects.models import (
    SharepointPreviewTool,
    SharepointGroundingToolParameters,
    ToolProjectConnection,
)

sharepoint_tool = SharepointPreviewTool(
    sharepoint_grounding_preview=SharepointGroundingToolParameters(
        project_connections=[
            ToolProjectConnection(
                project_connection_id=sharepoint_connection_id
            )
        ]
    )
)
```

### 8.3 Configure the MCP tool

The `MCPTool` connects to external data sources like Microsoft Learn documentation:

```python
from azure.ai.projects.models import MCPTool

mcp_tool = MCPTool(
    server_url="https://learn.microsoft.com/api/mcp",
    server_label="Microsoft_Learn_Documentation",
    require_approval="always",
)
```

### 8.4 Create the agent with tools

```python
agent = project_client.agents.create_version(
    agent_name="Modern Workplace Assistant",
    definition=PromptAgentDefinition(
        model="gpt-4o-mini",
        instructions=instructions,
        tools=[sharepoint_tool, mcp_tool],
    ),
)
```

### 8.5 Batch Evaluation with Built-in Evaluators

The evaluation framework uses `openai_client.evals` to run scalable, repeatable evaluations in the cloud:

| Evaluator | What it measures | Scale |
|---|---|---|
| `builtin.violence` | Violent or harmful content detection | 0-7 severity (lower = safer) |
| `builtin.fluency` | Response quality and readability | 1-5 |
| `builtin.task_adherence` | Whether the agent followed instructions | 1-5 |

In [ ]:
# Example: Running the enterprise agent tutorial
# Navigate to the tutorial directory and run:

import os
tutorial_path = "foundry-samples-tutorial/samples/python/enterprise-agent-tutorial/1-idea-to-prototype"

if os.path.exists(tutorial_path):
    print(f"Tutorial found at: {tutorial_path}")
    print("\nTo run the tutorial:")
    print(f"  1. cd {tutorial_path}")
    print(f"  2. Create .env with your FOUNDRY_PROJECT_ENDPOINT and FOUNDRY_MODEL_NAME")
    print(f"  3. pip install -r requirements.txt")
    print(f"  4. python main.py       # Run the agent")
    print(f"  5. python evaluate.py   # Run batch evaluation")
else:
    print(f"Tutorial not found. Run the git clone cell above first.")
    print(f"Or clone manually:")
    print(f"  git clone --depth 1 https://github.com/microsoft-foundry/foundry-samples.git")

---
## Part 9: Ask AI — The Portal AI Assistant

The [Foundry portal](https://ai.azure.com) includes a built-in AI assistant called **Ask AI**. Click the AI icon in the top-right bar to open a chat window.

### What Ask AI can do

| Capability | Description |
|---|---|
| **Documentation** | Navigate Foundry docs, find quickstarts, how-tos, and SDK reference |
| **Model Catalog** | Get info about specific models, their capabilities and features |
| **Troubleshooting** | Diagnose and resolve common Foundry problems |
| **Quota & Model Ops** | Deploy models, debug deployments, check quota/capacity by region |
| **Model Analysis** | Recommend models based on cost, performance, or quality; compare benchmarks |
| **Monitoring Insights** | Interpret evaluation dashboards, identify patterns and anomalies |
| **Evaluation Management** | Set up, execute, and monitor evaluation jobs |

### What Ask AI cannot do

- Answer questions outside of Foundry scope
- Call external APIs (only a specific subset of Foundry APIs)
- Bypass your permissions — it can only do what you can do
- Retain conversation history across sessions

### Actions & approvals

When Ask AI proposes actions that modify your Azure resources, it shows an approval flow. You can configure approval settings via the settings icon in the chat box:
- **System access** actions are pre-approved by default
- You can change settings anytime; they persist across sessions

> The Ask AI experience relies on the [Foundry MCP Server](https://learn.microsoft.com/en-us/azure/foundry/mcp/available-tools) and follows the same [security best practices](https://learn.microsoft.com/en-us/azure/foundry/mcp/security-best-practices).

---
## Part 10: Migrating from Foundry (classic)

If you're coming from the classic portal (Azure AI Studio / Azure AI Foundry), here's what changed and how to migrate.

### Key Migration Dates

| Date | Event |
|---|---|
| **May 30, 2026** | `azure-ai-inference` package retires — migrate to `openai` package |
| **August 26, 2026** | Assistants API sunsets — migrate to Responses API (Agents v2) |

### Terminology Mapping

| Concept | Classic | Current |
|---|---|---|
| Portal | Foundry (classic) | Foundry portal (toggle in banner) |
| Settings | Management Center | Operate section |
| Resource type | Azure OpenAI + Hub | Foundry Resource (single `AIServices` kind) |
| AI services | Azure AI Services | Foundry Tools |
| Model billing | Model-as-a-Service (MaaS) | Foundry Direct Models |
| API protocol | Assistants API | Responses API |
| API versioning | Monthly `api-version` params | v1 stable routes (no version param) |
| Conversation state | Threads | Conversations |
| Chat messages | Messages | Items (superset of messages) |
| Execution | Runs (async, polled) | Responses (sync by default) |
| Agent definition | Assistants / Agents | Agent Versions |
| Agent creation | `create_agent()` | `create_version()` |
| Endpoints | Multiple (5+ endpoints) | Single project endpoint |

### SDK Migration

| Current Package | Replaces | Notes |
|---|---|---|
| `openai` | `azure-ai-inference` | For model inference; `azure-ai-inference` retires May 2026 |
| `OpenAI()` with `base_url` | `AzureOpenAI()` | Standard client, no Azure-specific code |
| `azure-ai-projects` 2.x | `azure-ai-projects` 1.x | 2.x targets new portal; 1.x targets classic |
| `azure-ai-projects` 2.x | `azure-ai-generative`, `azure-ai-ml` | Capabilities merged into project client |

### SDK Migration Example

**Classic (before):**
```python
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint="https://my-resource.openai.azure.com",
    api_key="my-key",
    api_version="2024-12-01-preview"
)
```

**Current (after):**
```python
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

client = OpenAI(
    base_url="https://my-project.services.ai.azure.com/openai/v1",
    default_headers={
        "Authorization": f"Bearer {get_bearer_token_provider(
            DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default'
        )()}"
    }
)
```

### Portal Navigation Mapping

The classic portal uses a single left pane. The current portal splits features across **five top-level sections**:

| Section | Scope | What you find |
|---|---|---|
| **Home** | Selected project | Project overview and quick actions |
| **Discover** | Selected project | Model catalog and benchmarks |
| **Build** | Selected project | Agents, models, playgrounds, evaluations, fine-tuning |
| **Operate** | All projects | Admin, quota, compliance, fleet health, tracing |
| **Docs** | N/A | Documentation links |

| Task | Classic Location | Current Location |
|---|---|---|
| View deployments | Models + endpoints | **Build** > **Models** |
| Open playground | Playgrounds | **Build** > **Models** > select model |
| Build agents | Agents | **Build** > **Agents** |
| Model catalog | Model catalog | **Discover** > **Model catalog** |
| Evaluations | Evaluation | **Build** > **Evaluations** |
| Fine-tuning | Fine-tuning | **Build** > **Fine-tuning** |
| Tracing | Tracing | **Operate** > **Tracing** |
| Quotas | Management center > Quota | **Operate** > **Quota** |
| Users | Management center > Users | **Operate** > **Admin** |

### New Features (current portal only)

| Feature | Status |
|---|---|
| Responses API | GA |
| Agents v2 (Responses API) | GA |
| Tool catalog (1,400+ tools) | GA |
| Multi-agent workflows | Preview |
| Agent memory | Preview |
| Agent publishing to M365/Teams | GA |
| Foundry IQ | Preview |
| Hosted agents | Preview |
| A2A protocol | Preview |
| Foundry Control Plane | Preview |

> **Tip:** You can switch between classic and current portal anytime using the **New Foundry** toggle in the top banner. The toggle preserves your current project context.

---
## Troubleshooting

| Symptom | Cause | Resolution |
|---|---|---|
| `DefaultAzureCredential` auth error | Azure CLI session expired or not signed in | Run `az login` and retry |
| `Model deployment not found` | Model name in `.env` doesn't match a deployment | Check **Build > Models** in the Foundry portal, update `.env` |
| `403 Forbidden` on SharePoint | Insufficient permissions | Confirm your identity has Read access to the SharePoint library |
| MCP tool timeout | Microsoft Learn MCP server unreachable | Verify `MCP_SERVER_URL` is `https://learn.microsoft.com/api/mcp` and HTTPS is allowed |
| `ModuleNotFoundError` or unexpected API behavior | SDK version mismatch | Ensure `azure-ai-projects>=2.0.0` (2.x for new portal, 1.x for classic) |
| Projects missing in new portal | Hub-based projects not visible | Switch to classic portal or migrate to Foundry projects |
| Endpoint connection failures | Old multi-endpoint URLs | Update to single project endpoint format |
| Agent code returns `404` | Assistants API calls sent to Responses API endpoint | Rewrite to use Responses API (`create_version()` not `create_agent()`) |
| Agents unavailable in portal | Resource in unsupported region for Responses API | Create a Foundry resource in a [supported region](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses#region-availability) |

---
## Clean Up Resources

When you no longer need the resources, delete the resource group to remove everything:

In [ ]:
# WARNING: This deletes ALL resources in the resource group!
# Only run this when you're done with everything.

# !az group delete --name my-foundry-rg --yes --no-wait
print("Cleanup command is commented out for safety. Uncomment to run.")

---
## Summary

In this notebook you learned:

- [x] How to create Foundry resources (resource group, Foundry resource, project) via Azure CLI
- [x] How to deploy a model (gpt-4.1-mini) and verify the deployment
- [x] How to get your project endpoint and configure environment variables
- [x] How to grant team access using RBAC roles
- [x] How to chat with a model using the Responses API (`AIProjectClient` + `OpenAI()`)
- [x] How to create a versioned agent with `PromptAgentDefinition`
- [x] How to have multi-turn conversations using conversations
- [x] Enterprise patterns: SharePoint tools, MCP tools, and batch evaluation
- [x] How to use Ask AI in the Foundry portal
- [x] How to migrate from Foundry (classic) — terminology, SDKs, and portal navigation

### What's Next?

In the **next notebook**, we'll cover:
- Building a full Chainlit application with the agent
- Deploying to Azure App Service
- Configuring Managed Identity and IAM roles
- Testing the deployed application end-to-end

### Useful Links

- [Microsoft Foundry Portal](https://ai.azure.com)
- [Quickstart: Set up Foundry resources](https://learn.microsoft.com/en-us/azure/foundry/tutorials/quickstart-create-foundry-resources)
- [Get Started with Code](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code)
- [Developer Journey: Idea to Prototype](https://learn.microsoft.com/en-us/azure/foundry/tutorials/developer-journey-idea-to-prototype)
- [Ask AI](https://learn.microsoft.com/en-us/azure/foundry/concepts/ask-ai)
- [Migrate from Classic](https://learn.microsoft.com/en-us/azure/foundry/how-to/navigate-from-classic)
- [Foundry Models Overview](https://learn.microsoft.com/en-us/azure/foundry/concepts/foundry-models-overview)
- [Azure AI Projects SDK (PyPI)](https://pypi.org/project/azure-ai-projects/)
- [Foundry Samples Repository](https://github.com/microsoft-foundry/foundry-samples)